# Lecture 5 Studio — Computer Arithmetic
**Numerical Scientific Computing — M.Sc. Computer Engineering, Spring 2026**

Topics covered:
1. Integer representation (unsigned, sign-magnitude, two's complement, offset binary)
2. Floating-point representation (IEEE 754)
3. Special values: `Inf` and `NaN`
4. Overflow and Horner's method
5. Machine precision
6. Cancellation and rounding effects
7. Finite difference approximation & the imaginary trick
8. Function condition number

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['font.size'] = 12

---
## 1  Integer Representation

A computer stores everything as bits. How those bits are *interpreted* determines what number they represent.

### 1.1  Unsigned integers — limits

In [ ]:
for bits, dtype in [(8, np.uint8), (16, np.uint16), (32, np.uint32)]:
    info = np.iinfo(dtype)
    print(f"uint{bits:2d}:  max = {info.max:>15,}  (= 2^{bits} - 1 = {2**bits - 1:,})")

### 1.2  Sign-magnitude

In sign-magnitude the **MSB is the sign bit** (0 = positive, 1 = negative); the remaining bits give the magnitude directly.

| bit-pattern | unsigned value | sign-magnitude value |
|---|---|---|
| `0000 0000` | 0 | +0 |
| `0000 1010` | 10 | +10 |
| `1000 0000` | 128 | −0 |
| `1000 1010` | 138 | −10 |

#### Why is equality testing complex?

Zero has **two distinct bit-patterns**: `0000 0000` (+0) and `1000 0000` (−0).  
A naive bitwise comparison `a == b` returns `false` for these two patterns even though they represent the same mathematical value. Correct equality therefore requires special-casing: *"are both bit-patterns zero?"* must be checked separately before the bitwise test.

#### Why is addition/subtraction complex?

The algorithm depends on the **combination of signs** of the operands — you cannot simply add the bit-patterns. Four distinct cases arise:

| Operation | What you must do |
|---|---|
| `+a + +b` | Add magnitudes, result is positive |
| `−a − −b` = `−a + −b` | Add magnitudes, result is negative |
| `+a + −b` (a ≥ b) | Subtract magnitudes, result is positive |
| `+a + −b` (a < b) | Subtract magnitudes, result is negative |

The hardware must first **compare the magnitudes**, choose the right operation, then **assign the correct sign** — requiring a comparator, a subtractor, and branching logic on top of the adder.

In **two's complement** none of this is needed: the same unsigned adder handles all cases without any sign logic.

In [ ]:
def sign_magnitude_decode(bitstring: str) -> int:
    """Decode an n-bit sign-magnitude string (MSB = sign)."""
    sign = -1 if bitstring[0] == '1' else +1
    magnitude = int(bitstring[1:], 2)
    return sign * magnitude

def sign_magnitude_equal(a: str, b: str) -> bool:
    """Correct equality check for sign-magnitude: handles the +0 / -0 ambiguity."""
    # Naive bitwise check
    naive = (a == b)
    # Correct check: both are zero if magnitude bits are all 0
    a_is_zero = int(a[1:], 2) == 0
    b_is_zero = int(b[1:], 2) == 0
    correct = (a_is_zero and b_is_zero) or (a == b)
    return naive, correct

print("Decoding sign-magnitude bit-patterns:")
for b in ['00000000', '10000000', '00001010', '10001010']:
    print(f"  {b}  ->  {sign_magnitude_decode(b):+d}")

print("\nEquality testing — the +0 / -0 problem:")
pairs = [('00000000', '10000000'), ('00000000', '00000000'), ('00001010', '10001010')]
for a, b in pairs:
    naive, correct = sign_magnitude_equal(a, b)
    print(f"  {a} == {b}  |  naive (bitwise): {str(naive):<5}  |  correct: {correct}"
          f"  ({sign_magnitude_decode(a):+d} vs {sign_magnitude_decode(b):+d})")

print("\nArithmetic complexity — software must handle signs manually:")
def sign_magnitude_add(a: int, b: int) -> int:
    """Emulate sign-magnitude addition with all required case distinctions."""
    if a >= 0 and b >= 0:                      # case 1: both positive
        return a + b
    elif a < 0 and b < 0:                      # case 2: both negative
        return -(abs(a) + abs(b))
    elif abs(a) >= abs(b):                     # case 3/4: mixed signs
        return (1 if a >= 0 else -1) * (abs(a) - abs(b))
    else:
        return (1 if b >= 0 else -1) * (abs(b) - abs(a))

for a, b in [(5, 3), (-5, -3), (5, -3), (-5, 3), (3, -5)]:
    print(f"  sign_magnitude_add({a:+d}, {b:+d}) = {sign_magnitude_add(a, b):+d}")

### 1.3  Two's complement — the industry standard

Two's complement is the universal method for representing signed integers in modern hardware.  
The core idea: **a negative number is represented as the value you must add to its positive counterpart to produce zero** (with the carry bit discarded).

#### How to negate a number (two steps):
1. **Invert all bits** (flip every 0 to 1 and vice versa)
2. **Add 1**

Example — representing −5 in 8 bits:
```
 +5  =  0000 0101
invert  1111 1010
 + 1  -----------
 −5  =  1111 1011
```

**Verify:** +5 + (−5) must equal 0:
```
  0000 0101   (+5)
+ 1111 1011   (−5)
-----------
1 0000 0000   ← carry bit is discarded → result = 0  ✓
```

#### Reading a two's complement number:
- **Leading bit = 0** → positive; read the bit-pattern as a normal unsigned integer.
- **Leading bit = 1** → negative; its value is $-2^n$ plus the remaining bits.

Example: `1111 1110` in 8 bits = $-128 + 64 + 32 + 16 + 8 + 4 + 2 + 0 = -2$

#### Range for $n$ bits:

| | Value |
|---|---|
| Most positive | $2^{n-1} - 1$ |
| Most negative | $-2^{n-1}$ |
| Zero | unique — only `000...0` |

For 8 bits: **−128 to +127**. The range is asymmetric because zero occupies one slot on the positive side.

#### Why it wins over sign-magnitude:
- **Single zero** → equality is a simple bitwise comparison, no special cases.
- **Arithmetic is free** → the same unsigned adder circuit handles all sign combinations; no comparator or branching logic needed.
- **Leading bit is still a sign indicator** → easy to read.

In [ ]:
def twos_complement_negate(bitstring: str) -> str:
    """Negate a two's complement number: invert all bits, then add 1."""
    n = len(bitstring)
    inverted = ''.join('1' if b == '0' else '0' for b in bitstring)
    result = bin(int(inverted, 2) + 1)[2:].zfill(n)[-n:]  # keep only n bits
    return result

def twos_complement_decode(bitstring: str) -> int:
    n = len(bitstring)
    val = int(bitstring, 2)
    if bitstring[0] == '1':       # leading 1 → negative: value = -2^n + remaining
        val -= 2**n
    return val

print("Step-by-step negation (invert + add 1):")
for original in ['00000101', '00000011', '01111111', '10000000']:
    negated = twos_complement_negate(original)
    print(f"  {original} ({twos_complement_decode(original):+4d})  "
          f"-->  {negated} ({twos_complement_decode(negated):+4d})")

print("\nBoundary values for 8-bit two's complement:")
for b in ['00000000', '01111111', '10000000', '11111111']:
    print(f"  {b}  ->  {twos_complement_decode(b):+5d}")

print("\nVerify: addition works identically to unsigned arithmetic (carry discarded)")
for a, b in [(5, -5), (3, -5), (-3, -4), (127, 1)]:
    ba = bin(a % 256)[2:].zfill(8)
    bb = bin(b % 256)[2:].zfill(8)
    raw_sum = int(ba, 2) + int(bb, 2)
    result_bits = bin(raw_sum % 256)[2:].zfill(8)   # discard carry
    result = twos_complement_decode(result_bits)
    carry = raw_sum >> 8
    print(f"  {a:+4d} + {b:+4d}  =  {result:+4d}   "
          f"(carry={carry}, bits: {ba} + {bb} = {result_bits})")

print("\nnumpy int8 limits:", np.iinfo(np.int8).min, "to", np.iinfo(np.int8).max)
print(f"np.int8(127) + 1 = {np.int8(127) + np.int8(1)}  <- overflow wraps to -128")

### 1.4  Offset binary — used for the *exponent* field in IEEE 754

In offset binary (also called *biased* representation), a number is stored as:

$$\text{stored} = \text{true value} + B$$

where $B$ is a fixed **bias**. To recover the true value: $\text{true value} = \text{stored} - B$.

#### Key properties:

| Property | Detail |
|---|---|
| **Unique zero** | Stored value $B$ (e.g. `1000 0000` for 8-bit, bias=128) represents zero — unlike sign-magnitude |
| **Consistent ordering** | Larger bit-patterns always mean larger values → comparisons work with unsigned hardware |
| **Inverted sign bit** | The leading bit is `1` for non-negative and `0` for negative — opposite of two's complement |
| **Subtraction anomaly** | $(a+B) - (a+B) = 0$, but stored value `0` means the *most negative* number, not zero → unsuitable for general arithmetic |

#### Why IEEE 754 uses it for the exponent:

When comparing two floating-point numbers you want the one with the **larger exponent to sort higher**.  
Offset binary achieves this naturally: the stored exponent bit-patterns are ordered exactly like the true exponent values, so the entire float (sign + exponent + mantissa) can be compared as a single unsigned integer — no special floating-point comparator needed.

Biases used in IEEE 754:
- **Binary-16**: bias = 15
- **Binary-32**: bias = 127  
- **Binary-64**: bias = 1023

In [ ]:
bias64 = 1023
bias32 = 127

print("Stored exponent -> true exponent (Binary-64, bias=1023)")
for stored in [0, 1, 512, 1023, 1024, 2046]:
    true_exp = stored - bias64
    print(f"  stored={stored:4d}  ->  true exponent = {true_exp:+5d}")

print("\n(Stored=0 and stored=2047 are reserved for special values in Binary-64)")

---
## 2  Floating-Point Representation — IEEE 754

Every floating-point number is: $x = (-1)^s \cdot m \cdot 2^e$

| Format | Sign bits | Exponent bits | Mantissa bits |
|---|---|---|---|
| float16 (B16) | 1 | 5 | 10 |
| float32 (B32) | 1 | 8 | 23 |
| float64 (B64) | 1 | 11 | 52 |

In [ ]:
for dtype in [np.float16, np.float32, np.float64]:
    fi = np.finfo(dtype)
    print(f"{str(dtype.__name__):8s}  bits={fi.bits:3d}  "
          f"max={fi.max:.4e}  tiny={fi.tiny:.4e}  eps={fi.eps:.4e}")

### 2.1  Peeking inside: the raw bit layout of a float32

In [ ]:
def float32_bits(x: float) -> str:
    """Return the 32-bit IEEE 754 representation as a labelled string."""
    packed = struct.pack('>f', np.float32(x))
    bits = ''.join(f'{byte:08b}' for byte in packed)
    return f"sign={bits[0]}  exp={bits[1:9]} ({int(bits[1:9],2)})  mantissa={bits[9:]}"

for val in [1.0, -1.0, 0.5, 6.022e23, 0.0]:
    print(f"{val:>12g}  ->  {float32_bits(val)}")

### 2.2  Non-uniform spacing: floats are denser near zero

`numpy.nextafter` returns the next representable value — letting us see how the spacing varies.

In [ ]:
bases = [1e-3, 1e0, 1e3, 1e6, 1e12]
print("Value              Spacing to next float64")
for b in bases:
    spacing = np.nextafter(b, np.inf) - b
    rel = spacing / b
    print(f"  {b:.1e}    gap = {spacing:.4e}   (relative: {rel:.4e})")

---
## 3  Special Values: `Inf` and `NaN`

IEEE 754 reserves certain bit patterns for infinity and not-a-number.

In [ ]:
inf = np.float64(np.inf)
nan = np.float64(np.nan)

print("--- Infinity ---")
print(f"  overflow (large * large):      {np.float64(1.8e308) * 10}")
print(f"  division by zero:              {np.float64(1.0) / 0.0}")
print(f"  inf + inf:                     {inf + inf}")
print(f"  inf * 2:                       {inf * 2}")
print(f"  inf > 1e308:                   {inf > 1e308}")

print("\n--- NaN ---")
print(f"  inf - inf:                     {inf - inf}")
print(f"  0 * inf:                       {np.float64(0.0) * inf}")
print(f"  0 / 0:                         {np.float64(0.0) / 0.0}")
print(f"  inf / inf:                     {inf / inf}")
print(f"  sqrt(-1):                      {np.sqrt(np.float64(-1.0))}")
print(f"  nan == nan:                    {nan == nan}   <- NaN is not equal to itself!")
print(f"  np.isnan(nan):                 {np.isnan(nan)}")

---
## 4  Overflow Example: Polynomial Evaluation

Polynomial:
$$y = a_0 + a_1 x + a_2 x^2 + a_3 x^3 + a_4 x^4$$

Using **float16** (max $\approx 6.55 \times 10^4$), the sequential form overflows at intermediate steps.  
**Horner's method** restructures the computation to avoid this:
$$y = a_0 + x\bigl(a_1 + x\bigl(a_2 + x(a_3 + x\,a_4)\bigr)\bigr)$$

In [ ]:
coeffs = np.array([1200, -150, 11, -101, 12])   # a0..a4
x = np.float16(10.0)

# --- Sequential evaluation ---
steps_seq = []
result_seq = np.float16(0.0)
for n, a in enumerate(coeffs):
    term = np.float16(a) * x**n
    result_seq = result_seq + term
    steps_seq.append(float(result_seq))
    print(f"  After adding a{n}*x^{n} = {float(term):>10.2f}  ->  running sum = {float(result_seq)}")

print(f"\nSequential result (float16): {float(result_seq)}")

In [ ]:
# --- Horner's method ---
y = np.float16(0.0)
for a in reversed(coeffs):
    y = np.float16(a) + x * y
    print(f"  step: {float(y):.4g}")

print(f"\nHorner result    (float16): {float(y)}")

# Reference: exact value in float64
x64 = 10.0
y64 = sum(a * x64**n for n, a in enumerate(coeffs))
print(f"Reference        (float64): {y64}")

Horner's method avoids large intermediate values and **stays within the float16 range** while the sequential form hits $-\infty$.

---
## 5  Machine Precision

Machine precision $\varepsilon_2$ is the largest relative rounding error:
$$\varepsilon_2 \approx 2^{-p}$$
where $p$ is the number of mantissa bits.

In [ ]:
print(f"{'Format':<10} {'bits p':>8} {'eps2 = 2^(-p)':>18} {'numpy eps (=2*eps2)':>22}")
print("-" * 62)
for dtype, p in [(np.float16, 10), (np.float32, 23), (np.float64, 52)]:
    eps2_theory = 2**(-p)
    eps_np = np.finfo(dtype).eps          # numpy eps = 2 * eps2
    print(f"{dtype.__name__:<10} {p:>8}   {eps2_theory:>18.6e}   {eps_np:>22.6e}")

### 5.1  Empirical verification

We can *find* machine epsilon experimentally: start from 1, halve until `1 + eps == 1`.

In [ ]:
for dtype in [np.float16, np.float32, np.float64]:
    one = dtype(1.0)
    eps = dtype(1.0)
    while one + eps != one:
        eps_prev = eps
        eps = eps / dtype(2.0)
    print(f"{dtype.__name__:<10}  empirical eps2 = {float(eps_prev):.6e}  "
          f"  numpy eps/2 = {np.finfo(dtype).eps/2:.6e}")

---
## 6  Numerical Issues

### 6.1  Cancellation effect

When two nearly equal numbers are subtracted, *significant digits cancel*, leaving mostly rounding noise.

Classic example: $1 - \cos(x)$ near $x = 0$.

The numerically stable equivalent is: $2\sin^2(x/2)$.

In [ ]:
xs = np.logspace(-1, -14, 300)   # from 1e-1 down to 1e-14

# Reference in quad precision via mpmath
try:
    import mpmath
    mpmath.mp.dps = 50
    ref = np.array([float(1 - mpmath.cos(xi)) for xi in xs])
except ImportError:
    # Fallback: stable formula as reference
    ref = 2 * np.sin(xs / 2)**2

naive   = 1.0 - np.cos(xs)           # naive: cancellation for small x
stable  = 2 * np.sin(xs / 2)**2      # stable reformulation

rel_naive  = np.abs(naive  - ref) / (np.abs(ref) + 1e-300)
rel_stable = np.abs(stable - ref) / (np.abs(ref) + 1e-300)

fig, ax = plt.subplots()
ax.loglog(xs, rel_naive,  label=r'$1 - \cos(x)$  (naive)')
ax.loglog(xs, rel_stable, label=r'$2\sin^2(x/2)$ (stable)')
ax.invert_xaxis()
ax.set_xlabel('x')
ax.set_ylabel('Relative error')
ax.set_title('Cancellation effect in $1 - \cos(x)$')
ax.legend()
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.show()

### 6.2  Rounding accumulation

Summing the same value repeatedly amplifies rounding — the order and method of summation matter.

In [ ]:
N = 10_000
val = np.float32(0.1)

# Naive sequential sum in float32
s_naive = np.float32(0.0)
for _ in range(N):
    s_naive += val

# numpy built-in sum (uses pairwise summation)
s_numpy = np.sum(np.full(N, val, dtype=np.float32))

true_val = 0.1 * N   # = 1000.0

print(f"True value            : {true_val}")
print(f"Naive sequential sum  : {float(s_naive):.10f}  "
      f"(rel error = {abs(float(s_naive)-true_val)/true_val:.3e})")
print(f"numpy pairwise sum    : {float(s_numpy):.10f}  "
      f"(rel error = {abs(float(s_numpy)-true_val)/true_val:.3e})")

---
## 7  Finite Difference Approximation & the Imaginary Trick

Function: $y(x) = 1 - x^2 + 3x^3 - 5x^4 + 4x^5$

True derivative: $y'(x) = -2x + 9x^2 - 20x^3 + 20x^4$

### Central difference approximation:
$$y'_{\text{approx}}(x_0;\, x_\delta) = \frac{y(x_0+x_\delta) - y(x_0-x_\delta)}{2\,x_\delta}$$

This has a **sweet spot**: too large a step gives truncation error; too small a step causes cancellation as $y(x_0+x_\delta)$ and $y(x_0-x_\delta)$ become nearly equal.

### The imaginary trick — and why it works:

For any analytic function, applying a Taylor expansion with a purely imaginary step $j\varepsilon$:

$$y(x_0 + j\varepsilon) = \underbrace{y(x_0) - \frac{\varepsilon^2}{2}y''(x_0) + \cdots}_{\text{Re part}} + j\underbrace{\left(\varepsilon\, y'(x_0) - \frac{\varepsilon^3}{6}y'''(x_0) + \cdots\right)}_{\text{Im part}}$$

Dividing the imaginary part by $\varepsilon$:
$$\frac{\mathrm{Im}\bigl[y(x_0 + j\varepsilon)\bigr]}{\varepsilon} = y'(x_0) + O(\varepsilon^2)$$

**The key insight — no cancellation is possible:**

| Method | What gets subtracted | Risk |
|---|---|---|
| Central difference | $y(x_0+x_\delta) - y(x_0-x_\delta)$ | Both $\to y(x_0)$ as $x_\delta \to 0$ → catastrophic cancellation |
| Imaginary trick | Nothing — just extract `Im[...]` | $y(x_0)$ lives in the real part; $y'(x_0)$ lives in the imaginary part — stored in **separate registers**, never subtracted |

Because $y(x_0)$ and $y'(x_0)$ are encoded in orthogonal parts of the complex number, they cannot interfere. The error decreases as $O(\varepsilon^2)$ all the way to machine precision, with no cancellation floor.

In [ ]:
def y(x):
    return 1 - x**2 + 3*x**3 - 5*x**4 + 4*x**5

def dy_true(x):
    return -2*x + 9*x**2 - 20*x**3 + 20*x**4

def dy_central(x0, xd):
    return (y(x0 + xd) - y(x0 - xd)) / (2 * xd)

def dy_imag(x0, eps):
    return np.imag(y(x0 + 1j * eps)) / eps

x0 = 1.0
xd_values = np.logspace(-16, 0, 300)

true_deriv = dy_true(x0)
rel_central = np.abs(dy_central(x0, xd_values) - true_deriv) / np.abs(true_deriv)
rel_imag    = np.abs(dy_imag(x0, xd_values)    - true_deriv) / np.abs(true_deriv)

fig, ax = plt.subplots()
ax.loglog(xd_values, rel_central, label='Central difference')
ax.loglog(xd_values, rel_imag,    label='Imaginary trick', linestyle='--')
ax.axvline(x0 * np.sqrt(np.finfo(float).eps), color='grey',
           linestyle=':', label=r'Optimal $x_\delta \approx x_0\sqrt{\varepsilon_2}$')
ax.set_xlabel(r'Step size $x_\delta$')
ax.set_ylabel('Relative error')
ax.set_title(r'Derivative approximation of $y(x)$ at $x_0=1$')
ax.legend()
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.show()

**Observations:**
- Central difference shows a U-shape: large step → truncation error; small step → cancellation error.
- The imaginary trick avoids cancellation entirely — error decreases monotonically as $x_\delta \to 0$.

### 8.1  Analytical example: $f(x) = e^{kx}$

Jacobian: $J(x_0) = k\,e^{kx_0}$, so the condition number simplifies to $\kappa = k\,x_0$.

We fix $x_0 = 2$, $k = 10$ and sweep $x_\delta$ logarithmically from $10^{-6}$ to $10^{-1}$, comparing the numerical estimate against the analytical value.

In [ ]:
k  = 10.0
x0 = 2.0

def f_exp(x):
    return np.exp(k * x)

def kappa_analytical(x0):
    """kappa = |k * x0| for f(x) = exp(k*x)"""
    return np.abs(k * x0)

def kappa_numerical(x0, xd):
    return (np.abs(f_exp(x0 + xd) - f_exp(x0)) / np.abs(f_exp(x0))) * (np.abs(x0) / np.abs(xd))

xd_values = np.logspace(-6, -1, 300)   # x_delta from 1e-6 to 1e-1

kappa_num_sweep = kappa_numerical(x0, xd_values)
kappa_ana       = kappa_analytical(x0)          # scalar — independent of x_delta

fig, ax = plt.subplots()
ax.semilogx(xd_values, kappa_num_sweep,
            label=r'Numerical $\kappa(x_0, x_\delta)$', color='tab:red',
            marker='*', markevery=30, linewidth=1.5)
ax.axhline(kappa_ana, color='tab:green', linewidth=2,
           label=f'Analytical $\\kappa = k\\,x_0 = {kappa_ana:.1f}$')
ax.set_xlabel(r'Step size $x_\delta$')
ax.set_ylabel(r'$\kappa$')
ax.set_title(f'Condition number of $f(x)=e^{{kx}}$ at $x_0={x0}$, $k={k}$')
ax.legend()
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Analytical kappa  at x0={x0}:              {kappa_ana:.4f}")
print(f"Numerical  kappa  at x0={x0}, xd=1e-6:  {kappa_numerical(x0, 1e-6):.4f}")
print(f"Numerical  kappa  at x0={x0}, xd=1e-1:  {kappa_numerical(x0, 1e-1):.4f}")

### 8.2  An extremely ill-conditioned function (Rump's example)

$$f(x_0, x_1) = (333.75 - x_0^2)\,x_1^6 + x_0^2(11\,x_0^2 x_1^2 - 121\,x_1^4 - 2) + 5.5\,x_1^8 + \frac{x_0}{2\,x_1}$$

At $(x_0, x_1) = (77617, 33096)$: $f \approx -0.8274$ but nearly every finite-precision evaluation gives the wrong answer.

In [ ]:
def rump(x0, x1):
    return ((333.75 - x0**2) * x1**6
            + x0**2 * (11 * x0**2 * x1**2 - 121 * x1**4 - 2)
            + 5.5 * x1**8
            + x0 / (2 * x1))

a, b = 77617.0, 33096.0

print("Evaluating Rump's function at (77617, 33096)")
print(f"  float32 : {rump(np.float32(a), np.float32(b))}")
print(f"  float64 : {rump(np.float64(a), np.float64(b))}")

try:
    import mpmath
    print("\n  mpmath results (exact to k decimal places):")
    for k in [20, 30, 34, 36, 40]:
        mpmath.mp.dps = k
        r = rump(mpmath.mpf(a), mpmath.mpf(b))
        print(f"    k={k:2d}  ->  {float(r):.6f}")
except ImportError:
    print("\n  (install mpmath for multi-precision comparison: pip install mpmath)")

### 8.3  Estimating the condition number of Rump's function numerically

In [ ]:
x0_vec = np.array([a, b])
xd = 1e-4    # perturbation size
f0 = rump(*x0_vec)

# Numerical gradient via central differences
grad = np.zeros(2)
for i in range(2):
    dx = np.zeros(2)
    dx[i] = xd
    grad[i] = (rump(*(x0_vec + dx)) - rump(*(x0_vec - dx))) / (2 * xd)

# Condition number approximation (p=2 norm)
kappa_numerical_rump = np.linalg.norm(grad, 2) * np.linalg.norm(x0_vec, 2) / abs(f0)
print(f"Numerical condition number estimate: kappa_2 ~ {kappa_numerical_rump:.3e}")
print("(Lecture states the true kappa_2 = 5.3e37 — the numerical estimate is unreliable")
print(" at standard double precision because f0 itself is wrong!)") 

---
## Summary

| Topic | Key take-away |
|---|---|
| Integer representation | Two's complement dominates; sign-magnitude has two zeros |
| Floating-point | Sign + biased exponent + (implicit) mantissa; only $\mathbb{F} \subset \mathbb{R}$ |
| Special values | `Inf` from overflow/div-by-zero; `NaN` from undefined ops |
| Overflow / Horner | Order of operations matters — intermediate results can overflow |
| Machine precision | $\varepsilon_2 \approx 2^{-p}$; double gives $\approx 10^{-16}$ |
| Cancellation | Subtracting near-equal values destroys significant digits |
| Finite difference | Sweet spot around $x_\delta \approx x_0\sqrt{\varepsilon_2}$; imaginary trick avoids cancellation |
| Condition number | Quantifies sensitivity; high $\kappa$ means results are unreliable in finite precision |